# 1. Data Collection and ETF Universe Definition

## Overview
This notebook covers the first step in creating a DIY ETF portfolio: collecting and organizing ETF data. We'll:
1. Define our investment universe based on GDP and MSCI classification (equities & bonds)
2. Scrape ETF data from JustETF across four asset classes: **equities**, **bonds**, **precious metals**, and **commodities**
3. Verify ETF availability on our chosen platform

Equity and bond ETFs are scraped per country/region; precious metals and commodities are scraped globally (single universe, no regional breakdown).

## Required Libraries and Setup

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json

import pandas as pd
import numpy as np
import justetf_scraping

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.database import save_raw_etf_data

## Define Investment Universe
We'll start by creating a DataFrame of major economies, including their:
- GDP (2023 estimates)
- MSCI market classification (Developed/Emerging)
- Geographic region
- Country code for data collection

This regional universe applies to **equities** and **bonds** only. Precious metals and commodities ETCs are sourced from a single global screen.

In [3]:
# Create the country DataFrame using list of lists


country_data = [
    ["United States", "$27.721 trillion", "Developed", "AmericasandUK", "US", "USD"],
    ["Canada", "$2.142 trillion", "Developed", "AmericasandUK", "CA", "CAD"],
    ["United Kingdom", "$3.381 trillion", "Developed", "AmericasandUK", "GB", "GBP"],
    ["Japan", "$4.204 trillion", "Developed", "APAC", "JP", "JPY"],
    ["Australia", "$1.728 trillion", "Developed", "APAC", "AU", "AUD"],
    ["Germany", "$4.526 trillion", "Developed", "EMEA", "DE", "EUR"],
    ["France", "$3.052 trillion", "Developed", "EMEA", "FR", "EUR"],
    ["Italy", "$2.301 trillion", "Developed", "EMEA", "IT", "EUR"],
    ["Spain", "$1.620 trillion", "Developed", "EMEA", "ES", "EUR"],
    ["Netherlands", "$1.154 trillion", "Developed", "EMEA", "NL", "EUR"],
    ["Switzerland", "$884.94 billion", "Developed", "EMEA", "CH", "CHF"],
    ["Brazil", "$2.174 trillion", "Emerging", "Americas", "BR", "BRL"],
    ["Mexico", "$1.789 trillion", "Emerging", "Americas", "MX", "MXN"],
    ["China", "$17.795 trillion", "Emerging", "APACandEMEA", "CN", "CNY"],
    ["India", "$3.568 trillion", "Emerging", "APACandEMEA", "IN", "INR"],
    ["South Korea", "$1.713 trillion", "Emerging", "APACandEMEA", "KR", "KRW"],
    ["Indonesia", "$1.371 trillion", "Emerging", "APACandEMEA", "ID", "IDR"]
]

#Add currency to the each country


columns = ["Country", "2023 GDP", "MSCI", "Region", "Short_name", "Currency"]
country_df = pd.DataFrame(country_data, columns=columns)


#sort by MSCI then region
country_df.sort_values(by=["MSCI", "Region"], inplace=True)

## ETF Data Collection
Now we'll collect ETF data for all four asset classes:

- **Equities & Bonds** — scraped per country/region using the investment universe defined above
- **Precious Metals & Commodities** — scraped once globally (no regional breakdown)

Each scrape result is saved as a CSV in  and inserted into the  table in the SQLite database.

In [ ]:
import traceback
import warnings

# Define asset classes to collect
# class-preciousMetals and class-commodities are scraped globally (no country loop)
asset_classes = [
    "class-equity",
    "class-bonds",
    "class-preciousMetals",
    "class-commodities",
]

# --- Global-only asset classes (no country/currency iteration) ---
GLOBAL_ASSET_CLASSES = {"class-preciousMetals", "class-commodities"}

# Group countries by region for data organization
region_countries = country_df.groupby('Region')['Short_name'].apply(list).to_dict()

# Create a reverse mapping of country to region
country_to_region = dict(zip(country_df['Short_name'], country_df['Region']))
country_to_country_name = dict(zip(country_df['Short_name'], country_df['Country']))
country_to_msci = dict(zip(country_df['Short_name'], country_df['MSCI']))

# Create a mapping of MSCI and Region to Category
msci_region_to_category = {
    ('Developed', 'AmericasandUK'): 'Developed_AmericasandUK',
    ('Developed', 'EMEA'): 'Developed_EMEA',
    ('Developed', 'APAC'): 'Developed_APAC',
    ('Emerging', 'Americas'): 'Emerging_Americas',
    ('Emerging', 'APACandEMEA'): 'Emerging_APACandEMEA'
}

for asset_class in asset_classes:
    asset_class_name = asset_class.split("-", 1)[1]  # e.g. 'equity', 'bonds', 'preciousMetals'

    # ------------------------------------------------------------------
    # Global-only asset classes: single scrape, no country iteration
    # ------------------------------------------------------------------
    if asset_class in GLOBAL_ASSET_CLASSES:
        try:
            df = justetf_scraping.load_overview(asset_class=asset_class, local_country="GB")
            df['region'] = 'Global'
            df['country'] = 'Global'
            df['region_category'] = 'Global'

            filename = f'justetf_class-{asset_class_name}_global.csv'
            filepath = DATA_RAW / filename
            df.to_csv(filepath, index=False)
            print(f"Scraped {len(df)} ETFs for {asset_class_name} (Global) → {filename}")
        except Exception as e:
            print(f"Error scraping {asset_class} (Global): {e}")
            continue

        try:
            save_raw_etf_data(df, asset_class_name, 'Global')
        except Exception as e:
            warnings.warn(f"[DB] Could not save {asset_class_name}/Global to database: {e}")

        continue  # skip the country loop below

    # ------------------------------------------------------------------
    # Country/currency iteration for equity and bonds
    # ------------------------------------------------------------------
    for country in country_df['Short_name']:
        # --- Scrape + CSV save (fatal on failure) ---
        try:
            if asset_class == "class-equity":
                df = justetf_scraping.load_overview(asset_class=asset_class, country=country, local_country="GB")
            else:
                currency = country_df[country_df['Short_name'] == country]['Currency'].values[0]
                df = justetf_scraping.load_overview(asset_class=asset_class, instrument_currency=currency, local_country="GB")

            df['region'] = country_to_region[country]
            df['country'] = country_to_country_name[country]

            msci = country_to_msci[country]
            region = country_to_region[country]
            region_category = msci_region_to_category.get((msci, region), "Unknown")
            df['region_category'] = region_category

            # Save CSV (append + dedup by ticker)
            filename = f'justetf_{asset_class}_{region_category.lower()}.csv'.lower()
            filepath = DATA_RAW / filename

            if filepath.exists():
                existing_df = pd.read_csv(filepath)
                combined_df = pd.concat([existing_df, df], ignore_index=True).drop_duplicates(subset=['ticker'])
                combined_df.to_csv(filepath, index=False)
            else:
                df.to_csv(filepath, index=False)

            print(f"Processed {country} data for {asset_class}")
        except Exception as e:
            print(f"Error scraping {asset_class} for {country}: {e}")
            continue

        # --- DB write (non-fatal: CSV is the fallback) ---
        try:
            save_raw_etf_data(df, asset_class_name, region_category)
        except Exception as e:
            warnings.warn(f"[DB] Could not save {asset_class_name}/{region_category} to database: {e}")


In [5]:
country_to_country_name

{'JP': 'Japan',
 'AU': 'Australia',
 'US': 'United States',
 'CA': 'Canada',
 'GB': 'United Kingdom',
 'DE': 'Germany',
 'FR': 'France',
 'IT': 'Italy',
 'ES': 'Spain',
 'NL': 'Netherlands',
 'CH': 'Switzerland',
 'CN': 'China',
 'IN': 'India',
 'KR': 'South Korea',
 'ID': 'Indonesia',
 'BR': 'Brazil',
 'MX': 'Mexico'}

## Data Collection Summary
Display summary of collected ETFs by asset class and region:

In [6]:
asset_classes, country_df

(['class-equity', 'class-bonds'],
            Country          2023 GDP       MSCI         Region Short_name  \
 3            Japan   $4.204 trillion  Developed           APAC         JP   
 4        Australia   $1.728 trillion  Developed           APAC         AU   
 0    United States  $27.721 trillion  Developed  AmericasandUK         US   
 1           Canada   $2.142 trillion  Developed  AmericasandUK         CA   
 2   United Kingdom   $3.381 trillion  Developed  AmericasandUK         GB   
 5          Germany   $4.526 trillion  Developed           EMEA         DE   
 6           France   $3.052 trillion  Developed           EMEA         FR   
 7            Italy   $2.301 trillion  Developed           EMEA         IT   
 8            Spain   $1.620 trillion  Developed           EMEA         ES   
 9      Netherlands   $1.154 trillion  Developed           EMEA         NL   
 10     Switzerland   $884.94 billion  Developed           EMEA         CH   
 13           China  $17.795 t

In [ ]:
# Summarize collected data â€” iterate over files in data/raw/
summary_data = []
for filepath in sorted(DATA_RAW.glob("justetf_class-*.csv")):
    csvl = filepath.name
    asset_class = get_asset_class_from_filename(csvl)
    region_category = get_region_category_from_filename(csvl)
    df = pd.read_csv(filepath)
    summary_data.append({
        'Asset Class': asset_class.title(),
        'Category': region_category,
        'NumberofETFs': len(df),
        'DistributingETFs': len(df[df['dividends'] == 'Distributing'])
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df.pivot_table(
    index='Category',
    columns='Asset Class',
    values=['NumberofETFs', 'DistributingETFs'],
    aggfunc='first'
))

In [43]:
summary_df

,Asset Class,Region,Number of ETFs,Distributing ETFs
0,Equity,APAC,208,54
1,Equity,Americas,468,141
2,Equity,EMEA,143,66
3,Bonds,APAC,35,14
4,Bonds,Americas,178,94
5,Bonds,EMEA,63,42
